# Lista 4 — Modelo de Câmera

Resolução de [Tasks/lista_4.pdf](Tasks/lista_4.pdf). Execute as células em ordem no Jupyter ou no Google Colab. Cada gráfico abre com os valores do enunciado e pode ser alterado pelos controles. As figuras são salvas em PNG no diretório de execução.

##### Setup

###### 1. Importações

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.interpolate import CubicSpline, make_interp_spline

###### 2. Funções compartilhadas de exibição

In [ ]:
def exibir_controles(funcao, controles):
    """Executa o exemplo imediatamente e atualiza o gráfico ao mudar controles."""
    saida = widgets.interactive_output(funcao, controles)
    display(widgets.VBox(list(controles.values())), saida)


def configurar_plano(eixo, titulo):
    eixo.set(xlabel="x", ylabel="y", title=titulo)
    eixo.set_aspect("equal", adjustable="box")
    eixo.grid(True, color="#dddddd", linewidth=0.8)
    eixo.set_axisbelow(True)
    eixo.margins(0.12)


def finalizar_figura(figura, arquivo):
    figura.tight_layout()
    figura.savefig(arquivo, dpi=160, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(figura)


# Compartilhada pelas questões 3 e 6.
def curva_bezier(pontos, amostras=100):
    """Avalia a curva cúbica de Bézier nos quatro pontos Nx2."""
    pontos = np.asarray(pontos, dtype=float)
    if pontos.shape != (4, 2):
        raise ValueError("A curva cúbica precisa de quatro pontos 2D")
    t = np.linspace(0, 1, amostras)[:, None]
    return ((1-t)**3 * pontos[0] + 3*t*(1-t)**2 * pontos[1]
            + 3*t**2*(1-t) * pontos[2] + t**3 * pontos[3])

##### QUESTÃO 1 — Polilinhas

O gráfico inicial usa vértices aproximados da imagem de referência para desenhar um Fusca de perfil com polilinhas. A opção **Exemplo do PDF** conserva exatamente os nove vértices impressos, que sozinhos formam um losango. Casa, estrela e coração também usam somente segmentos de reta. A escala inicial é `1`, e o carro é normalizado para uma largura próxima de `8`, como no exemplo do PDF.

In [ ]:
# Coordenadas aproximadas da imagem de referência (origem da imagem no alto à esquerda).
# Cada linha abaixo é uma polilinha: contorno, rodas, janelas, porta ou detalhe.
def plano_da_imagem(pontos):
    pontos = np.asarray(pontos, dtype=float)
    return np.column_stack(((pontos[:, 0] - 35) / 44.25,
                            (175 - pontos[:, 1]) / 44.25))


def circulo_em_segmentos(cx, cy, raio, quantidade=24):
    angulos = np.linspace(0, 2*np.pi, quantidade+1)
    return np.column_stack((cx + raio*np.cos(angulos),
                            cy + raio*np.sin(angulos)))


fusca_imagem = [
    np.array([(37, 127), (44, 113), (55, 99), (76, 91), (101, 83),
              (128, 77), (151, 73), (163, 57), (174, 44), (199, 37),
              (230, 33), (258, 35), (284, 41), (308, 51), (338, 76),
              (359, 95), (376, 117), (389, 139)], dtype=float),
    np.array([(37, 127), (35, 139), (56, 139)], dtype=float),
    np.array([(149, 143), (270, 143)], dtype=float),
    np.array([(355, 143), (389, 143), (389, 139)], dtype=float),
    np.array([(56, 143), (62, 125), (76, 113), (93, 107),
              (110, 110), (128, 122), (140, 139), (149, 143)], dtype=float),
    np.array([(270, 143), (280, 122), (296, 109), (319, 105),
              (338, 112), (350, 128), (355, 143)], dtype=float),
    circulo_em_segmentos(92, 145, 34),
    circulo_em_segmentos(92, 145, 23),
    circulo_em_segmentos(319, 145, 34),
    circulo_em_segmentos(319, 145, 23),
    np.array([(151, 82), (172, 46), (181, 45), (181, 82), (151, 82)], dtype=float),
    np.array([(181, 45), (204, 44), (232, 44), (232, 82),
              (181, 82)], dtype=float),
    np.array([(235, 45), (259, 47), (281, 53), (299, 63),
              (310, 74), (299, 83), (235, 83), (235, 45)], dtype=float),
    np.array([(151, 82), (299, 82)], dtype=float),
    np.array([(232, 44), (232, 151), (270, 151)], dtype=float),
    np.array([(149, 143), (149, 151), (232, 151)], dtype=float),
    np.array([(219, 91), (236, 91)], dtype=float),
    np.array([(54, 97), (79, 91), (104, 85), (150, 78)], dtype=float),
    np.array([(338, 76), (340, 79), (310, 79)], dtype=float),
    np.array([(35, 127), (32, 128)], dtype=float),
    np.array([(389, 139), (390, 146), (375, 146)], dtype=float),
]
fusca = [plano_da_imagem(pontos) for pontos in fusca_imagem]

formas = {
    "Fusca": fusca,
    "Exemplo do PDF": [np.array([(0, 0), (2, 1), (4, 2), (6, 1),
                                  (8, 0), (6, -1), (4, -2), (2, -1),
                                  (0, 0)], dtype=float)],
    "Casa": [np.array([(0, 0), (0, 3), (2, 5), (4, 3),
                        (4, 0), (0, 0)], dtype=float)],
    "Estrela": [np.array([(0, 3), (1, 0), (4, 0), (1.6, -1.8),
                          (2.6, -4.5), (0, -2.8), (-2.6, -4.5),
                          (-1.6, -1.8), (-4, 0), (-1, 0), (0, 3)], dtype=float)],
    "Coração": [np.array([(0, -3), (-1.5, -1.5), (-3, 0.5), (-3, 2),
                           (-2, 3), (-1, 3), (0, 2), (1, 3), (2, 3),
                           (3, 2), (3, 0.5), (1.5, -1.5), (0, -3)], dtype=float)],
}


def mostrar_polilinha(forma, escala):
    figura, eixo = plt.subplots(figsize=(10, 4.5))
    for pontos in formas[forma]:
        vertices = pontos * escala
        eixo.plot(vertices[:, 0], vertices[:, 1], "-o",
                  color="#dc8a1d" if forma == "Fusca" else "tab:blue",
                  linewidth=1.3 if forma == "Fusca" else 2,
                  markersize=1.8 if forma == "Fusca" else 5)
    if forma == "Fusca":
        eixo.set_aspect("equal", adjustable="box")
        eixo.set_xlim(-0.35*escala, 8.3*escala)
        eixo.set_ylim(-0.3*escala, 3.5*escala)
        eixo.axis("off")
    else:
        configurar_plano(eixo, f"Questão 1 — Polilinha: {forma}")
    finalizar_figura(figura, "lista4_questao1_polilinha.png")


exibir_controles(mostrar_polilinha, {
    "forma": widgets.Dropdown(options=list(formas), value="Fusca", description="Forma"),
    "escala": widgets.FloatSlider(value=1, min=0.5, max=2, step=0.1,
                                  description="Escala", continuous_update=False),
})

##### QUESTÃO 2 — Spline cúbica

Os cinco pontos iniciais são os do PDF: `x = [0, 2, 4, 6, 8]` e `y = [0, 1, 2, 1, 0]`. O controle acrescenta pontos interpolados com uma pequena variação, para observar como a curva se ajusta.

In [ ]:
def mostrar_spline_cubica(pontos_extras, altura):
    x_base = np.array([0, 2, 4, 6, 8], dtype=float)
    y_base = np.array([0, 1, 2, 1, 0], dtype=float)
    if pontos_extras:
        x_extra = np.array([1, 3, 5, 7], dtype=float)[:pontos_extras]
        y_extra = np.interp(x_extra, x_base, y_base) + 0.25 * np.sin(np.pi * x_extra / 4)
        x = np.concatenate((x_base, x_extra))
        y = np.concatenate((y_base, y_extra))
        ordem = np.argsort(x)
        x, y = x[ordem], y[ordem]
    else:
        x, y = x_base, y_base
    y = y * altura
    spline = CubicSpline(x, y)
    t = np.linspace(0, 8, 200)
    figura, eixo = plt.subplots(figsize=(8, 4))
    eixo.plot(x, y, "o", label="Pontos de controle")
    eixo.plot(t, spline(t), "-", label="Spline cúbica")
    configurar_plano(eixo, "Questão 2 — Curva suave do Fusca")
    eixo.legend()
    finalizar_figura(figura, "lista4_questao2_spline.png")


exibir_controles(mostrar_spline_cubica, {
    "pontos_extras": widgets.IntSlider(value=0, min=0, max=4, step=1,
                                        description="Pontos extras", continuous_update=False),
    "altura": widgets.FloatSlider(value=1, min=0.5, max=2, step=0.1,
                                   description="Altura", continuous_update=False),
})

##### QUESTÃO 3 — Curva de Bézier

Os quatro pontos de controle começam em `(0,0)`, `(2,3)`, `(4,3)` e `(6,0)`. As alturas dos controles internos alteram o para-lama.

In [ ]:
def mostrar_bezier(y1, y2):
    pontos = np.array([[0, 0], [2, y1], [4, y2], [6, 0]], dtype=float)
    curva = curva_bezier(pontos)
    figura, eixo = plt.subplots(figsize=(7, 4.5))
    eixo.plot(pontos[:, 0], pontos[:, 1], "o--", label="Pontos de controle")
    eixo.plot(curva[:, 0], curva[:, 1], "-", linewidth=2.5, label="Curva de Bézier")
    configurar_plano(eixo, "Questão 3 — Para-lama do Fusca")
    eixo.legend()
    finalizar_figura(figura, "lista4_questao3_bezier.png")


exibir_controles(mostrar_bezier, {
    "y1": widgets.FloatSlider(value=3, min=-2, max=6, step=0.25,
                              description="P1: y", continuous_update=False),
    "y2": widgets.FloatSlider(value=3, min=-2, max=6, step=0.25,
                              description="P2: y", continuous_update=False),
})

##### QUESTÃO 4 — B-Spline cúbica

Os nove pontos do teto e o grau `k=3` são os do PDF. Pontos adicionais são amostrados entre os originais para experimentar maior densidade de controle.

In [ ]:
def mostrar_bspline(pontos_extras, altura):
    x_base = np.arange(9, dtype=float)
    y_base = np.array([0, 1, 2, 2.5, 2.8, 2.5, 2, 1, 0], dtype=float)
    if pontos_extras:
        x_extra = np.arange(pontos_extras, dtype=float) + 0.5
        x = np.sort(np.concatenate((x_base, x_extra)))
        y = np.interp(x, x_base, y_base)
    else:
        x, y = x_base, y_base
    y = y * altura
    spline = make_interp_spline(x, y, k=3)
    t = np.linspace(0, 8, 200)
    figura, eixo = plt.subplots(figsize=(8, 4))
    eixo.plot(x, y, "o", label="Pontos de controle")
    eixo.plot(t, spline(t), "-", label="B-Spline cúbica")
    configurar_plano(eixo, "Questão 4 — Teto do Fusca")
    eixo.legend()
    finalizar_figura(figura, "lista4_questao4_bspline.png")


exibir_controles(mostrar_bspline, {
    "pontos_extras": widgets.IntSlider(value=0, min=0, max=8, step=1,
                                        description="Pontos extras", continuous_update=False),
    "altura": widgets.FloatSlider(value=1, min=0.5, max=2, step=0.1,
                                   description="Altura", continuous_update=False),
})

##### QUESTÃO 5 — Poliedros e malhas 3D

Compare (a) esfera poligonal, (b) Suzanne, (c) superfície de quadriláteros e (d) superfície de triângulos. O controle de densidade altera (a), (c) e (d). Suzanne usa `Archives/suzanne.obj`; ao abrir apenas o notebook no Colab, o OBJ é baixado automaticamente da mesma fonte usada na lista 3.

In [ ]:
def carregar_suzanne():
    caminho = Path("Archives/suzanne.obj")
    if not caminho.is_file():
        caminho = Path("suzanne.obj")
    if not caminho.is_file():
        fonte = ("https://raw.githubusercontent.com/alecjacobson/"
                 "common-3d-test-models/master/data/suzanne.obj")
        with urlopen(fonte, timeout=30) as resposta:
            caminho.write_bytes(resposta.read())
    vertices, faces = [], []
    with caminho.open(encoding="utf-8") as arquivo:
        for linha in arquivo:
            if linha.startswith("v "):
                vertices.append([float(valor) for valor in linha.split()[1:4]])
            elif linha.startswith("f "):
                faces.append([int(parte.split("/")[0]) - 1
                              for parte in linha.split()[1:]])
    vertices = np.asarray(vertices, dtype=float)
    if not len(vertices) or not len(faces):
        raise ValueError("OBJ sem vértices ou faces")
    vertices -= (vertices.min(axis=0) + vertices.max(axis=0)) / 2
    return vertices, faces


vertices_suzanne, faces_suzanne = carregar_suzanne()


def malha_grade(n, superficie):
    """Gera vértices e faces quadradas ou triangulares de uma grade."""
    u = np.linspace(-1, 1, n)
    x, y = np.meshgrid(u, u)
    z = 0.32 * np.cos(np.pi*x) * np.cos(np.pi*y)
    vertices = np.column_stack((x.ravel(), y.ravel(), z.ravel()))
    faces = []
    for i in range(n-1):
        for j in range(n-1):
            a = i*n + j
            quad = [a, a+1, a+n+1, a+n]
            if superficie == "triangular":
                faces.extend(([quad[0], quad[1], quad[2]],
                              [quad[0], quad[2], quad[3]]))
            else:
                faces.append(quad)
    return vertices, faces


def malha_esfera(n):
    latitudes = np.linspace(0, np.pi, n+1)
    longitudes = np.linspace(0, 2*np.pi, 2*n+1)[:-1]
    vertices = np.array([[np.sin(t)*np.cos(p), np.sin(t)*np.sin(p), np.cos(t)]
                         for t in latitudes for p in longitudes])
    m = len(longitudes)
    faces = []
    for i in range(n):
        for j in range(m):
            a = i*m+j
            b = i*m+(j+1)%m
            c = (i+1)*m+(j+1)%m
            d = (i+1)*m+j
            if i > 0:
                faces.append([a, b, d])
            if i < n-1:
                faces.append([b, c, d])
    return vertices, faces


def mostrar_malhas(densidade, elevacao, azimute):
    malhas = [
        (*malha_esfera(densidade), "(a) Esfera poligonal"),
        (vertices_suzanne, faces_suzanne, "(b) Suzanne"),
        (*malha_grade(densidade, "quadrilateral"), "(c) Quadriláteros"),
        (*malha_grade(densidade, "triangular"), "(d) Triângulos"),
    ]
    figura = plt.figure(figsize=(12, 10))
    for indice, (vertices, faces, titulo) in enumerate(malhas, 1):
        eixo = figura.add_subplot(2, 2, indice, projection="3d")
        malha = Poly3DCollection([vertices[face] for face in faces],
                                 facecolor="#7db7d4", edgecolor="#34495e",
                                 linewidth=0.3, alpha=0.8)
        eixo.add_collection3d(malha)
        if indice != 2:
            eixo.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2],
                         color="tab:red", s=4)
        limite = np.max(np.abs(vertices)) * 1.15
        eixo.set(xlim=(-limite, limite), ylim=(-limite, limite),
                 zlim=(-limite, limite), title=titulo)
        eixo.set_box_aspect((1, 1, 1))
        eixo.view_init(elev=elevacao, azim=azimute)
    finalizar_figura(figura, "lista4_questao5_malhas.png")


exibir_controles(mostrar_malhas, {
    "densidade": widgets.IntSlider(value=5, min=3, max=12, step=1,
                                    description="Densidade", continuous_update=False),
    "elevacao": widgets.IntSlider(value=25, min=-30, max=80, step=5,
                                   description="Elevação", continuous_update=False),
    "azimute": widgets.IntSlider(value=-60, min=-180, max=180, step=5,
                                  description="Azimute", continuous_update=False),
})

##### QUESTÃO 6 — Projeto integrador: Curvas do Fusca

O corpo usa polilinhas; o capô e a traseira usam splines cúbicas; os para-lamas e a janela usam Bézier; o teto usa B-Spline cúbica. Ajuste altura, comprimento e curvatura para compor outras versões.

In [ ]:
def mostrar_fusca(altura_teto, comprimento, curvatura_lamas):
    figura, eixo = plt.subplots(figsize=(11, 5))
    # Polilinha do corpo: soleira, para-choques e contorno das caixas de roda.
    corpo = np.array([[0, 0.15], [0.5, 0.05], [1.1, 0.05],
                      [1.5, 0.1], [3.1, 0.1], [3.5, 0.05],
                      [5.2, 0.05], [5.6, 0.1], [7.3, 0.1],
                      [7.7, 0.15], [8, 0.25]])
    eixo.plot(corpo[:, 0], corpo[:, 1], color="#22313f", linewidth=3,
              label="Polilinha: corpo")

    # Splines cúbicas interpolam os perfis do capô e da traseira.
    for x, y in (([0, 0.5, 1.1, 1.8], [0.15, 0.65, 0.92, 1.12]),
                 ([6.2, 6.9, 7.5, 8], [1.2, 0.9, 0.55, 0.25])):
        t = np.linspace(x[0], x[-1], 100)
        eixo.plot(t, CubicSpline(x, y)(t), color="#3484a6", linewidth=3,
                  label="Spline: capô/traseira" if x[0] == 0 else None)

    # B-Spline cúbica do teto, com a mesma forma básica da questão 4.
    x_teto = np.array([1.8, 2.2, 2.8, 3.5, 4.2, 4.9, 5.5, 5.9, 6.2])
    y_teto = np.array([1.12, 1.55, 2.05, 2.35, 2.5, 2.42, 2.12, 1.7, 1.2])
    y_teto = 1.12 + (y_teto - 1.12) * altura_teto
    t = np.linspace(x_teto[0], x_teto[-1], 200)
    eixo.plot(t, make_interp_spline(x_teto, y_teto, k=3)(t),
              color="#d87636", linewidth=4, label="B-Spline: teto")

    # Bézier desenha para-lamas e linhas arredondadas das janelas.
    for centro in (2.25, 6.1):
        arco = curva_bezier([[centro-0.8, 0.1],
                            [centro-0.55, curvatura_lamas],
                            [centro+0.55, curvatura_lamas],
                            [centro+0.8, 0.1]])
        eixo.plot(arco[:, 0], arco[:, 1], color="#9b59b6", linewidth=3,
                  label="Bézier: para-lamas" if centro == 2.25 else None)
        roda = plt.Circle((centro, 0.05), 0.43, facecolor="#25313e",
                          edgecolor="#666", linewidth=2, zorder=4)
        eixo.add_patch(roda)
        eixo.add_patch(plt.Circle((centro, 0.05), 0.2,
                                   facecolor="#dadfe3", zorder=5))
    janela = curva_bezier([[2.7, 1.28], [3.1, 2.0*altura_teto],
                           [4.8, 2.0*altura_teto], [5.45, 1.28]])
    eixo.plot(janela[:, 0], janela[:, 1], color="#9b59b6",
              linewidth=2, label="Bézier: janela")
    eixo.plot([4.25, 4.25], [1.25, max(1.28, 2.0*altura_teto-0.12)],
              color="#22313f", linewidth=2)
    eixo.plot([1.1, 7.3], [0.95, 0.95], color="#22313f", linewidth=2)

    # Comprimento muda a escala horizontal de toda a composição.
    for linha in eixo.lines:
        linha.set_xdata(np.asarray(linha.get_xdata(), dtype=float) * comprimento)
    for roda in eixo.patches:
        roda.center = (roda.center[0] * comprimento, roda.center[1])
    eixo.set_xlim(-0.5, 8.5*comprimento)
    eixo.set_ylim(-0.5, max(3, 2.7*altura_teto))
    configurar_plano(eixo, "Questão 6 — Fusca estilizado")
    eixo.legend(loc="upper right", ncol=2, fontsize=8)
    finalizar_figura(figura, "lista4_questao6_fusca.png")


exibir_controles(mostrar_fusca, {
    "altura_teto": widgets.FloatSlider(value=1, min=0.7, max=1.3, step=0.05,
                                        description="Altura teto", continuous_update=False),
    "comprimento": widgets.FloatSlider(value=1, min=0.8, max=1.2, step=0.05,
                                        description="Comprimento", continuous_update=False),
    "curvatura_lamas": widgets.FloatSlider(value=0.9, min=0.5, max=1.5, step=0.05,
                                            description="Para-lamas", continuous_update=False),
})